In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time
import warnings
from memory_profiler import memory_usage

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

In [ ]:
# Carga del Conjunto Ciego (Test Blind Set)
TEST_PATH = 'data/test_blind_set.csv'

df_test = pd.read_csv(TEST_PATH, sep=';')
df_test['texto_lineal'] = df_test['texto_lineal'].fillna('')

print(f"Conjunto Ciego (Hold-out) cargado.")
print(f"   Total de registros : {len(df_test):,}")
print(f"   Distribucion de clases:\n{df_test['label'].value_counts().to_string()}")

y_true = df_test['label'].values

In [ ]:
def graficar_matriz(y_real, y_pred, titulo, ax):
    cm = confusion_matrix(y_real, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Falso (0)', 'Real (1)'],
                yticklabels=['Falso (0)', 'Real (1)'])
    ax.set_title(titulo, fontsize=12, pad=10)
    ax.set_ylabel('Etiqueta Real')
    ax.set_xlabel('Predicción')

In [ ]:
# Carga y Evaluación de Modelos Lineales
RUTA_SVM = 'data/modelo_optimizado_svm.joblib'
RUTA_LOG = 'data/modelo_optimizado_lr.joblib'

resultados_lineales = {}

def evaluar_modelo_lineal(ruta_modelo, nombre_modelo):
    if not os.path.exists(ruta_modelo):
        print(f"No se encontró el modelo: {ruta_modelo}")
        return None
    
    print(f"\nEvaluando: {nombre_modelo}")
    modelo = joblib.load(ruta_modelo)
    
    # Perfilado de memoria y tiempo en inferencia
    def _predict():
        return modelo.predict(df_test['texto_lineal'])
        
    t_start = time.time()
    muestras_ram, y_pred = memory_usage((_predict, [], {}), interval=0.01, include_children=True, max_usage=False, retval=True)
    t_end = time.time()
    
    t_inferencia = t_end - t_start
    pico_ram_mb = max(muestras_ram) - min(muestras_ram) # RAM adicional usada durante inferencia
    
    f1 = f1_score(y_true, y_pred, average='macro')
    
    print(f"  F1-Score Macro : {f1:.4f}")
    print(f"  Tiempo Infer.  : {t_inferencia:.4f} seg")
    print(f"  RAM Adicional  : {pico_ram_mb:.2f} MB")
    
    return {
        'pred': y_pred,
        'f1': f1,
        't_inferencia': t_inferencia,
        'ram_adicional': pico_ram_mb
    }

resultados_lineales['SVM'] = evaluar_modelo_lineal(RUTA_SVM, "SVM (LinearSVC)")
resultados_lineales['Regresión Logística'] = evaluar_modelo_lineal(RUTA_LOG, "Regresión Logística")

In [ ]:
# Visualización de Resultados
num_modelos = sum(1 for v in resultados_lineales.values() if v is not None)

if num_modelos > 0:
    fig, axes = plt.subplots(1, num_modelos, figsize=(5 * num_modelos, 4))
    if num_modelos == 1:
        axes = [axes]
        
    idx = 0
    resumen_lista = []
    
    for nombre, res in resultados_lineales.items():
        if res is not None:
            graficar_matriz(y_true, res['pred'], f"Matriz: {nombre}", axes[idx])
            idx += 1
            resumen_lista.append({
                'Modelo': nombre,
                'F1-Score Macro': round(res['f1'], 4),
                'Tiempo Inferencia (s)': round(res['t_inferencia'], 4),
                'RAM Adicional (MB)': round(res['ram_adicional'], 2)
            })

    plt.tight_layout()
    plt.show()
    
    resumen_df = pd.DataFrame(resumen_lista)
    display(resumen_df)
    resumen_df.to_csv('data/resultados_evaluacion_lineales.csv', sep=';', index=False)
    print("Métricas guardadas en: data/resultados_evaluacion_lineales.csv")